In [1]:
import sys
sys.path.insert(0, '..')
import requests
import pandas as pd
import time
from tools.wiki_scrapper import get_bond_mod_names

In [2]:
bond_mods_list = get_bond_mod_names()
print(bond_mods_list)

['aerial_bond', 'astral_bond', 'contagious_bond', 'covert_bond', 'duplex_bond', 'manifold_bond', 'momentous_bond', 'mystic_bond', 'reinforced_bond', 'restorative_bond', 'seismic_bond', 'tandem_bond', 'tenacious_bond', 'vicious_bond']


In [3]:
def get_order_request(url):
    time.sleep(1)
    response = requests.get(url)
    response.raise_for_status()
    return response

def get_bond_mod_offers(bond_mods_list: list[str]):
    df_concat = pd.DataFrame()
    for mod_name in bond_mods_list:
        print(f"Collecting orders from {mod_name}")
        url = f"https://api.warframe.market/v2/orders/item/{mod_name}"
        mod_orders = get_order_request(url)
        df_order = pd.DataFrame(mod_orders.json()["data"])
        df_concat = pd.concat([df_concat, df_order])
    df_concat['user'] = df_concat['user'].str['ingameName']
    df_concat['mod_name'] = mod_name
    df_concat = df_concat[['user', 'rank', 'platinum', 'type', 'quantity', 'perTrade', 'createdAt', 'updatedAt']]
    df_concat = df_concat.sort_values('rank', ascending=False)
    return df_concat

def get_bond_mod_timeseries(bond_mods_list:list[str]):
    collect_modes = ["statistics_closed", "statistics_live"]
    time_period = ["90days", "48hours"]
    df_concat = pd.DataFrame()
    for mod_name in bond_mods_list:
        print(f"Collecting timeseries from {mod_name}")
        url = f"https://api.warframe.market/v1/items/{mod_name}/statistics"
        mod_orders = get_order_request(url)
        for mode in collect_modes:
            for period in time_period:
                df_order = pd.DataFrame(mod_orders.json()["payload"][mode][period])
                df_order['period'] = period
                df_order["mode"] = ("live" if mode == "statistics_live" else "closed")
                df_order["mod_name"] = mod_name
                df_concat = pd.concat([df_concat, df_order])
    df_concat = df_concat.sort_values('datetime', ascending=False)
    return df_concat

In [4]:
orders_df = get_bond_mod_offers(bond_mods_list)
display(orders_df)

,user,rank,platinum,type,quantity,perTrade,createdAt,updatedAt
273,Spiriro40,5,24,sell,1,1,2026-09-25T08:49:29Z,2026-09-28T12:09:13Z
269,gyx_szd,5,20,sell,1,1,2026-09-24T06:24:28Z,2026-09-28T00:39:24Z
259,CollinsPrime404,5,13,sell,1,1,2026-09-21T14:14:46Z,2026-09-21T14:14:46Z
29,W_WW4,5,29,sell,1,1,2025-01-25T17:27:32Z,2026-08-03T19:07:49Z
28,gogohappy,5,25,sell,4,1,2025-01-03T07:06:31Z,2026-09-24T21:25:59Z
...,...,...,...,...,...,...,...,...
31,ZombieEPx,0,15,sell,1,1,2025-03-04T01:33:15Z,2026-09-20T16:21:51Z
30,hord3n,0,18,sell,6,1,2025-02-25T23:57:47Z,2026-09-04T15:50:48Z
20,Ares,0,50,sell,1,1,2024-07-10T07:50:20Z,2026-09-24T21:28:59Z
16,NEFFEXyyds,0,35,sell,20,1,2024-06-02T17:24:04Z,2026-08-15T08:56:40Z


In [5]:
orders_df.to_csv('../data/bond_orders.csv', sep=';')

In [6]:
timeseries_df = get_bond_mod_timeseries(bond_mods_list)
display(timeseries_df)

,datetime,volume,min_price,max_price,open_price,closed_price,avg_price,wa_price,median,moving_avg,donch_top,donch_bot,id,mod_rank,period,mode,mod_name,order_type
94,2026-09-28T12:00:00.000+00:00,33,15.0,25.0,NaN,NaN,20.0,20.818,20.0,NaN,NaN,NaN,6aba678ce63ff80010c54972,5,48hours,live,seismic_bond,sell
95,2026-09-28T12:00:00.000+00:00,64,13.0,20.0,NaN,NaN,16.5,14.891,15.0,NaN,NaN,NaN,6aba6789e63ff80010c54938,0,48hours,live,mystic_bond,sell
95,2026-09-28T12:00:00.000+00:00,65,14.0,20.0,NaN,NaN,17.0,16.585,16.0,NaN,NaN,NaN,6aba678be63ff80010c5495a,0,48hours,live,aerial_bond,sell
141,2026-09-28T12:00:00.000+00:00,900,6.0,6.0,NaN,NaN,6.0,6.000,6.0,NaN,NaN,NaN,6aba6787e63ff80010c54918,0,48hours,live,tenacious_bond,buy
55,2026-09-28T12:00:00.000+00:00,3,14.0,15.0,14.0,15.0,14.5,14.667,15.0,14.4,16.0,13.0,6aba6787e63ff80010c54914,0,48hours,closed,tenacious_bond,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,2026-07-01T00:00:00.000+00:00,26,3.0,5.0,NaN,NaN,4.0,4.385,5.0,NaN,NaN,NaN,6a45abb61e047a0010d4d97a,5,90days,live,contagious_bond,buy
1,2026-07-01T00:00:00.000+00:00,348,11.0,20.0,NaN,NaN,15.5,16.862,15.0,NaN,NaN,NaN,6a45abb51e047a0010d4d956,0,90days,live,tandem_bond,sell
0,2026-07-01T00:00:00.000+00:00,184,19.0,35.0,NaN,NaN,27.0,23.255,24.0,NaN,NaN,NaN,6a45abb51e047a0010d4d95f,5,90days,live,tandem_bond,sell
1,2026-07-01T00:00:00.000+00:00,584,13.0,17.0,NaN,NaN,15.0,14.755,15.0,NaN,NaN,NaN,6a45abb61e047a0010d4d971,0,90days,live,contagious_bond,sell


In [7]:
timeseries_df.to_csv('../data/bond_90days_orders.csv', sep=';')